In [ ]:
%%capture

!pip install boto3
!pip install unsloth
!pip install s3fs

In [ ]:
# Base libraries required
import os
import torch
import boto3
import pandas as pd
from io import StringIO
from datasets import load_dataset, Dataset

# Adding model libraries to train
from unsloth.chat_templates import train_on_responses_only
from unsloth import FastLanguageModel, is_bfloat16_supported
from trl import SFTTrainer
from transformers import TrainingArguments, DataCollatorForSeq2Seq

In [ ]:
model, tokenizer = FastLanguageModel.from_pretrained(
    model_name="unsloth/Llama-3.2-3B-Instruct",
    max_seq_length=2048,
    dtype=None,
    load_in_4bit=True,
)

model = FastLanguageModel.get_peft_model(
    model,
    r=64,
    target_modules=["q_proj", "k_proj", "v_proj", "o_proj",
                    "gate_proj", "up_proj", "down_proj",],
    lora_alpha=16,
    lora_dropout=0.1,
    bias="none",

    use_gradient_checkpointing="unsloth",
    random_state=3407,
    use_rslora=False,
    loftq_config=None,
)

In [ ]:
import os
s3client = boto3.client('s3', region_name='us-east-2',
                        aws_access_key_id = os.environ["AWS_ACCESS_KEY_ID"],
                        aws_secret_access_key = os.environ["AWS_SECRET_ACCESS_KEY"])

In [ ]:
# bucket = os.environ["S3_BUCKET"]
# s3Object = "KnowledgeBase/Processed/ConvDataFinalVibhorg.csv"
# local_path = "/content/ConvDataFinalVibhorg.csv"

# s3client.download_file(bucket, s3Object, local_path)

# conv_dataset = load_dataset("csv", data_files=local_path, split="train")

In [ ]:
bucket = os.environ["S3_BUCKET"]
s3Object = "KnowledgeBase/Processed/OLD_FILES/ConvDataFinalVibhorg.csv"
local_path = "/content/OldConvDataFinalVibhorg.csv"

s3client.download_file(bucket, s3Object, local_path)

conv_dataset = load_dataset("csv", data_files=local_path, split="train")

In [ ]:
conv_dataset

In [ ]:
# sampled_dataset = conv_dataset.select(range(2000))
# sampled_dataset

In [ ]:
SYSTEM_PROMPT = """You are a helpful and joyous mental health agent for student athletes of color named Alfreeda.
Always answer as helpfully and cheerfully as possible, while being safe. Your answers should not include any harmful, unethical, racist, sexist, toxic, dangerous, hateful, or illegal content.
Please ensure that your responses are socially unbiased and positive in nature. Don't respond to a prompt that is unrelated to mental health (positive or negative). Keep the conversation flowing. Use this context from previous conversation."""

In [ ]:
def format_chat_template(attribute):
    if attribute["History"] ==  None:
        attribute["History"] = ""

    json_format = [{"role": "system", "content": SYSTEM_PROMPT + attribute["History"]},
                   {"role": "user", "content": attribute["Context"]},
                   {"role": "assistant", "content": attribute["Response"]}]

    attribute["trainText"] = tokenizer.apply_chat_template(json_format,
                                                      tokenize=False)

    return attribute

final_conversations = conv_dataset.map(format_chat_template,
                                        num_proc=16)

final_conversations = final_conversations.train_test_split(test_size=0.1)

In [ ]:
trainer = SFTTrainer(
                        model,
                        tokenizer = tokenizer,
                        train_dataset = final_conversations["train"],
                        eval_dataset = final_conversations["test"],
                        dataset_text_field = "trainText",
                        max_seq_length = 2048,
                        data_collator = DataCollatorForSeq2Seq(tokenizer=tokenizer),
                        # dataset_num_proc = 16,
                        packing = False,
                        args = TrainingArguments(
                                                per_device_train_batch_size = 2,
                                                per_device_eval_batch_size = 2,
                                                gradient_accumulation_steps = 4,
                                                eval_strategy = "steps",
                                                eval_steps = 0.2,
                                                warmup_steps = 5,
                                                num_train_epochs = 2.0,
                                                learning_rate = 2e-4,
                                                fp16 = not is_bfloat16_supported(),
                                                bf16 = is_bfloat16_supported(),
                                                optim = "adamw_8bit",
                                                weight_decay = 0.01,
                                                lr_scheduler_type = "linear",
                                                seed = 3407,
                                                output_dir = "./Vib",
                                                report_to = "none",
                                            ),
                )

In [ ]:
trainer = train_on_responses_only(
                                    trainer,
                                    instruction_part="<|start_header_id|>user<|end_header_id|>\n\n",
                                    response_part="<|start_header_id|>assistant<|end_header_id|>\n\n",
                                )

In [ ]:
trainer_stats = trainer.train()

In [ ]:
user_input = "I am having stress with my work, can you help me with maintaining stress-relief"
history = "Context: Hello, how are you? My name is Manu. Response: Hi Manu, I'm good, thank you! How can I assist you today? Context: Give me a productivity tip! Response: Absolutely, I'd be happy to share one with you! One effective tip is to prioritize your tasks based on their importance and deadlines. Start with the most critical ones first and then move on to the less urgent ones. This helps you stay focused and manage your workload efficiently."

In [ ]:
user_input = "Give me a productivity tip!"
history = ""

In [ ]:
messages = [{"role": "system", "content": SYSTEM_PROMPT + history},
            {"role": "user", "content": user_input}]

prompt = tokenizer.apply_chat_template(messages, tokenize=False, add_generation_prompt=True)

inputs = tokenizer(prompt, return_tensors='pt', padding=True, truncation=True).to("cuda")

outputs = model.generate(**inputs, num_return_sequences=1) # max_new_tokens=150,

response = tokenizer.decode(outputs[0], skip_special_tokens=True)
prediction = response.split("assistant")[1].strip()

In [ ]:
response

In [ ]:
prediction